# LeetCode 3364 — Minimum Positive Sum Subarray bằng Python

Link đề: https://leetcode.com/problems/minimum-positive-sum-subarray/ · Độ khó: **Easy** · Chủ đề: Array (mảng), Prefix Sum (tổng tiền tố), Sliding Window (cửa sổ trượt)

Skeleton LeetCode:

```python
class Solution:
    def minimumSumSubarray(self, nums: List[int], l: int, r: int) -> int:
        ...
```

Ghi chú: đây là bài **Premium**, nên phần đề bài trong notebook được viết lại theo hiểu biết về bài; nếu bạn có tài khoản Premium, hãy đối chiếu lại đề gốc. Ba ví dụ chính lấy theo nguồn đề mà mình có; mọi đáp án đều được kiểm chứng bằng code (đối chiếu bốn cách làm và một bộ vét cạn độc lập trên nhiều input ngẫu nhiên). Các ví dụ tự dựng được ghi rõ là "tự dựng".

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho mảng `nums` và hai số `l`, `r`; trong tất cả **mảng con** (subarray, dãy liên tiếp) có **độ dài từ `l` đến `r`** và **tổng lớn hơn 0**, hãy trả về **tổng nhỏ nhất**; nếu không có mảng con nào thoả thì trả `-1`.
2. Nắm các công cụ Python: cắt lát `nums[i:j+1]`, hàm `sum`, so sánh dây chuyền `l <= length <= r`, giá trị vô cực `float("inf")` làm "lính canh", và `itertools.accumulate` để tạo mảng tổng tiền tố.
3. Đi từ cách **vét cạn ba vòng lặp** `O(n^3)` → **cộng dồn khi mở rộng đầu phải** `O(n^2)` → **tổng tiền tố** `S[j+1] - S[i]` → **cửa sổ trượt cho từng độ dài cố định**.
4. Hiểu mấu chốt: chỉ có **hai điều kiện lọc** (độ dài trong `[l, r]` và tổng `> 0`) rồi lấy `min`; với `n <= 100` ta thoải mái duyệt hết mọi cặp `(i, j)`, miễn là **không tính lại tổng từ đầu** mỗi lần.
5. Tránh các lỗi kinh điển: dùng `s >= 0` thay vì `s > 0`, quên kiểm tra độ dài, khởi tạo đáp án bằng `0`, trả `inf` thay vì `-1`, và nhầm `r` thành độ dài **nhỏ nhất** hay "chỉ số cuối".

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và **hai** số nguyên `l`, `r`. Nhiệm vụ: tìm **tổng nhỏ nhất** của một **mảng con** thoả **cả hai** điều kiện:

- độ dài của mảng con nằm trong đoạn `[l, r]` (gồm cả hai đầu mút);
- tổng của mảng con **lớn hơn 0**.

Nếu không có mảng con nào thoả, trả về `-1`. (**Mảng con** là một dãy phần tử **liên tiếp, không rỗng** của mảng.)

**Ví dụ 1**

- Input: `nums = [3, -2, 1, 4], l = 2, r = 3`
- Output: `1`
- Giải thích: các mảng con độ dài từ 2 đến 3 có tổng dương là `[3, -2]` (tổng 1), `[1, 4]` (tổng 5), `[3, -2, 1]` (tổng 2), `[-2, 1, 4]` (tổng 3). Nhỏ nhất là `1`. (Mảng con `[-2, 1]` có tổng `-1` nên bị loại.)

**Ví dụ 2**

- Input: `nums = [-2, 2, -3, 1], l = 2, r = 3`
- Output: `-1`
- Giải thích: không mảng con nào có độ dài `2..3` mà tổng dương.

**Ví dụ 3**

- Input: `nums = [1, 2, 3, 4], l = 2, r = 4`
- Output: `3`
- Giải thích: `[1, 2]` có độ dài 2 và tổng `3`, nhỏ nhất trong các tổng dương.

**Ràng buộc**

- `1 <= nums.length <= 100`
- `1 <= l <= r <= nums.length`
- `-1000 <= nums[i] <= 1000`

In [1]:
from typing import List

examples = [
    (([3, -2, 1, 4], 2, 3), 1),
    (([-2, 2, -3, 1], 2, 3), -1),
    (([1, 2, 3, 4], 2, 4), 3),
]

def min_pos_quick(nums, l, r):
    """Bản nhanh để chạy thử ví dụ: thử mọi cặp (i, j)."""
    best = None
    for i in range(len(nums)):
        for j in range(i, len(nums)):
            s = sum(nums[i:j + 1])
            if l <= j - i + 1 <= r and s > 0:
                if best is None or s < best:
                    best = s
    return -1 if best is None else best

for (nums, l, r), expected in examples:
    print(f"nums={nums}, l={l}, r={r} -> {min_pos_quick(nums, l, r)} (kỳ vọng {expected})")
    assert min_pos_quick(nums, l, r) == expected

nums=[3, -2, 1, 4], l=2, r=3 -> 1 (kỳ vọng 1)
nums=[-2, 2, -3, 1], l=2, r=3 -> -1 (kỳ vọng -1)
nums=[1, 2, 3, 4], l=2, r=4 -> 3 (kỳ vọng 3)


## 3. Kiến thức nền

### 3.1. Mảng con (subarray) và cắt lát `nums[i:j+1]`

**Mảng con** là một đoạn **liên tiếp** của mảng. Khác với "dãy con" (subsequence) có thể bỏ phần tử ở giữa. Với mảng `nums`, mảng con từ chỉ số `i` đến chỉ số `j` (gồm cả `j`) được lấy bằng **cắt lát** `nums[i:j+1]`. Chú ý: trong Python, `a[x:y]` lấy các chỉ số `x, x+1, ..., y-1`, tức là **không gồm `y`**, nên muốn gồm `j` ta phải viết `j + 1`. Độ dài của mảng con đó là `j - i + 1`.

In [2]:
nums = [3, -2, 1, 4]
i, j = 1, 3
print(nums[i:j + 1])          # [-2, 1, 4]
print("độ dài:", j - i + 1)   # 3
print(len(nums[i:j + 1]))     # 3 (cùng kết quả)

# Đoạn không liên tiếp KHÔNG phải mảng con: [3, 1] bỏ qua -2 nên chỉ là dãy con
print([nums[0], nums[2]])

[-2, 1, 4]
độ dài: 3
3
[3, 1]


### 3.2. Hàm `sum` và cộng dồn `s += x`

`sum(a)` cộng mọi phần tử của `a`. Với mỗi mảng con ta có thể gọi `sum(nums[i:j+1])` — nhưng làm vậy sẽ **cộng lại từ đầu** mỗi lần. Cách thông minh hơn là **cộng dồn**: giữ một biến `s`, mỗi lần mở rộng thêm một phần tử thì `s += nums[j]`. Đây chính là bước tối ưu đầu tiên của bài này.

In [3]:
nums = [3, -2, 1, 4]
print(sum(nums[0:3]))          # 3 + (-2) + 1 = 2

# cộng dồn: mở rộng đầu phải từ i = 0
s = 0
for j in range(0, 4):
    s += nums[j]
    print(f"j={j}: thêm {nums[j]:>2} -> tổng mảng con nums[0..{j}] = {s}")

2
j=0: thêm  3 -> tổng mảng con nums[0..0] = 3
j=1: thêm -2 -> tổng mảng con nums[0..1] = 1
j=2: thêm  1 -> tổng mảng con nums[0..2] = 2
j=3: thêm  4 -> tổng mảng con nums[0..3] = 6


### 3.3. So sánh dây chuyền `l <= length <= r`

Python cho viết `l <= length <= r` thay cho `l <= length and length <= r`. Nó đúng khi `length` **nằm trong đoạn** `[l, r]` (gồm hai đầu mút). Đây là cách ngắn gọn nhất để kiểm tra "độ dài hợp lệ".

In [4]:
l, r = 2, 3
for length in range(1, 6):
    print(f"length={length}: {l <= length <= r}")
# chỉ length = 2 và 3 là True

length=1: False
length=2: True
length=3: True
length=4: False
length=5: False


### 3.4. Giá trị "lính canh" `float("inf")`

Để tìm **giá trị nhỏ nhất**, ta cần một giá trị khởi đầu mà **mọi số thật đều nhỏ hơn**. `float("inf")` (vô cực dương) làm đúng việc đó: `min(inf, x) = x` với mọi số `x`. Sau vòng lặp, nếu `ans` vẫn là `inf` nghĩa là **chưa có mảng con nào hợp lệ** ⇒ trả `-1`.

**Đừng** khởi tạo `ans = 0`: khi đó `min(0, s)` luôn `<= 0` mà ta chỉ quan tâm tổng dương, nên ta sẽ không bao giờ cập nhật được.

In [5]:
inf = float("inf")
ans = inf
for s in [5, 2, 9]:
    ans = min(ans, s)
print(ans)             # 2

ans = inf
print(ans == inf)      # True  -> chưa có phần tử hợp lệ nào
print(-1 if ans == inf else ans)   # -1

# Sai: khởi tạo 0 làm mất mọi tổng dương
ans0 = 0
for s in [5, 2, 9]:
    ans0 = min(ans0, s)
print(ans0)            # 0 (sai, không phản ánh đáp án 2)

2
True
-1
0


### 3.5. Tổng tiền tố (prefix sum): `S[j+1] - S[i]`

Gọi `S[0] = 0`, `S[k+1] = S[k] + nums[k]`. Khi đó `S[k]` là tổng của `k` phần tử **đầu tiên**. Tổng đoạn `nums[i..j]` bằng `S[j+1] - S[i]` (lấy tổng `j+1` phần tử đầu, trừ đi tổng `i` phần tử đầu). Có `S` rồi thì **mọi tổng mảng con tính trong `O(1)`**. `itertools.accumulate(nums, initial=0)` tạo luôn mảng `S`.

In [6]:
from itertools import accumulate

nums = [3, -2, 1, 4]
S = list(accumulate(nums, initial=0))
print(S)                          # [0, 3, 1, 2, 6]
i, j = 1, 3
print(S[j + 1] - S[i])            # 6 - 3 = 3
print(sum(nums[i:j + 1]))         # 3 (khớp)

[0, 3, 1, 2, 6]
3
3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `nums = [3, -2, 1, 4]`, `l = 2`, `r = 3`: liệt kê **mọi** mảng con rồi lọc hai điều kiện.

| mảng con | chỉ số (i..j) | độ dài | tổng | độ dài trong [2, 3]? | tổng > 0? | giữ lại? |
|---|---|---|---|---|---|---|
| `[3]` | 0..0 | 1 | 3 | không | có | không |
| `[3, -2]` | 0..1 | 2 | 1 | có | có | **có** |
| `[3, -2, 1]` | 0..2 | 3 | 2 | có | có | **có** |
| `[3, -2, 1, 4]` | 0..3 | 4 | 6 | không | có | không |
| `[-2]` | 1..1 | 1 | -2 | không | không | không |
| `[-2, 1]` | 1..2 | 2 | -1 | có | không | không |
| `[-2, 1, 4]` | 1..3 | 3 | 3 | có | có | **có** |
| `[1]` | 2..2 | 1 | 1 | không | có | không |
| `[1, 4]` | 2..3 | 2 | 5 | có | có | **có** |
| `[4]` | 3..3 | 1 | 4 | không | có | không |

Các tổng được giữ lại: `1, 2, 3, 5`. Nhỏ nhất là **`1`**. Để ý hai "cái bẫy" của bài: `[3]` có tổng `3` nhưng **độ dài 1 < l** nên không được tính; `[-2, 1]` có độ dài hợp lệ nhưng tổng **không dương**.

In [7]:
def show_subarrays(nums, l, r):
    n = len(nums)
    kept = []
    for i in range(n):
        for j in range(i, n):
            length = j - i + 1
            s = sum(nums[i:j + 1])
            ok_len = l <= length <= r
            ok_sum = s > 0
            keep = ok_len and ok_sum
            if keep:
                kept.append(s)
            print(f"{str(nums[i:j + 1]):<16} i..j={i}..{j} len={length} sum={s:>3} len ok={ok_len!s:<5} sum>0={ok_sum!s:<5} giữ={keep}")
    print("các tổng giữ lại:", sorted(kept), "-> nhỏ nhất:", min(kept) if kept else -1)

show_subarrays([3, -2, 1, 4], 2, 3)

[3]              i..j=0..0 len=1 sum=  3 len ok=False sum>0=True  giữ=False
[3, -2]          i..j=0..1 len=2 sum=  1 len ok=True  sum>0=True  giữ=True
[3, -2, 1]       i..j=0..2 len=3 sum=  2 len ok=True  sum>0=True  giữ=True
[3, -2, 1, 4]    i..j=0..3 len=4 sum=  6 len ok=False sum>0=True  giữ=False
[-2]             i..j=1..1 len=1 sum= -2 len ok=False sum>0=False giữ=False
[-2, 1]          i..j=1..2 len=2 sum= -1 len ok=True  sum>0=False giữ=False
[-2, 1, 4]       i..j=1..3 len=3 sum=  3 len ok=True  sum>0=True  giữ=True
[1]              i..j=2..2 len=1 sum=  1 len ok=False sum>0=True  giữ=False
[1, 4]           i..j=2..3 len=2 sum=  5 len ok=True  sum>0=True  giữ=True
[4]              i..j=3..3 len=1 sum=  4 len ok=False sum>0=True  giữ=False
các tổng giữ lại: [1, 2, 3, 5] -> nhỏ nhất: 1


## 5. Cách 1 — Brute Force / baseline: ba vòng lặp, tính tổng bằng `sum`

**Ý tưởng.** Làm đúng như bảng ở mục 4: duyệt mọi điểm đầu `i`, mọi điểm cuối `j >= i`; tính độ dài và **tổng bằng `sum(nums[i:j+1])`**; nếu thoả hai điều kiện thì cập nhật `ans = min(ans, s)`. Cuối cùng nếu `ans` vẫn là `inf` thì trả `-1`.

In [8]:
def min_pos_brute(nums: List[int], l: int, r: int) -> int:
    n = len(nums)
    inf = float("inf")
    ans = inf
    for i in range(n):                    # điểm đầu
        for j in range(i, n):             # điểm cuối
            length = j - i + 1
            if l <= length <= r:
                s = sum(nums[i:j + 1])    # cộng lại từ đầu mỗi lần
                if s > 0:
                    ans = min(ans, s)
    return -1 if ans == inf else ans

for (nums, l, r), expected in examples:
    print(min_pos_brute(nums, l, r), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
-1 (kỳ vọng -1 )
3 (kỳ vọng 3 )


**Độ phức tạp (Cách 1).** Có `O(n^2)` cặp `(i, j)`, và mỗi cặp ta tốn thêm `O(n)` để cộng ⇒ **Time `O(n^3)`**, Space `O(n)` cho mỗi lát cắt tạm (hoặc `O(1)` nếu cộng bằng vòng lặp). Với `n = 100` thì `~10^6` phép cộng — vẫn chạy được, nhưng **lãng phí**.

**Nút thắt.** Khi `i` cố định và `j` tăng lên `j + 1`, mảng con mới chỉ **hơn mảng con cũ đúng một phần tử `nums[j+1]`**. Vậy mà `sum(...)` cộng lại từ đầu. Ta không cần làm thế.

## 6. Ý tưởng thuật toán / bước đột phá

**Cộng dồn khi mở rộng đầu phải.** Cố định `i`, cho `j` chạy từ `i` sang phải và giữ biến `s` là tổng `nums[i..j]`. Mỗi bước `j -> j + 1` chỉ cần `s += nums[j + 1]` — một phép cộng thay vì `O(n)`. Tổng số phép cộng là `O(n^2)`.

Có hai nhận xét nhỏ để code gọn hơn:

1. Điều kiện độ dài `l <= j - i + 1 <= r` có thể kiểm tra **ngay trong vòng lặp**; khi độ dài vượt `r` thì mọi `j` lớn hơn cũng vượt ⇒ có thể **dừng sớm** (`break`).
2. Chỉ cập nhật `ans` khi cả hai điều kiện đều đúng; **tổng không dương thì bỏ qua**, nhưng **không được dừng**, vì mở rộng thêm có thể làm tổng dương lên.

In [9]:
# Demo: cộng dồn từ i = 0 của [3, -2, 1, 4] với l = 2, r = 3
nums, l, r = [3, -2, 1, 4], 2, 3
i, s = 0, 0
for j in range(i, len(nums)):
    s += nums[j]
    length = j - i + 1
    print(f"j={j}, len={length}, s={s:>2}, hợp lệ độ dài? {l <= length <= r}, tổng>0? {s > 0}")

j=0, len=1, s= 3, hợp lệ độ dài? False, tổng>0? True
j=1, len=2, s= 1, hợp lệ độ dài? True, tổng>0? True
j=2, len=3, s= 2, hợp lệ độ dài? True, tổng>0? True
j=3, len=4, s= 6, hợp lệ độ dài? False, tổng>0? True


## 7. Các cách tiếp theo

### Cách 2 — Cộng dồn khi mở rộng đầu phải, có dừng sớm (lời giải chính thức)

**Giải thích từng đoạn.**

1. `ans = inf` — lính canh, chưa thấy tổng hợp lệ nào.
2. `for i in range(n):` — chọn điểm đầu; `s = 0` đặt lại tổng cho mỗi điểm đầu mới.
3. `for j in range(i, n):` — mở rộng đầu phải; `s += nums[j]` cập nhật tổng trong `O(1)`.
4. `length = j - i + 1`; `if length > r: break` (dừng sớm), `if length >= l and s > 0: ans = min(ans, s)`.
5. `return -1 if ans == inf else ans`.

In [10]:
def min_pos_running(nums: List[int], l: int, r: int) -> int:
    n = len(nums)
    inf = float("inf")
    ans = inf
    for i in range(n):
        s = 0
        for j in range(i, n):
            s += nums[j]
            length = j - i + 1
            if length > r:
                break                      # dài quá r: mở rộng tiếp cũng vô ích
            if length >= l and s > 0:
                ans = min(ans, s)
    return -1 if ans == inf else ans

for (nums, l, r), expected in examples:
    print(min_pos_running(nums, l, r), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
-1 (kỳ vọng -1 )
3 (kỳ vọng 3 )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
from typing import List

class SolutionRunning:
    def minimumSumSubarray(self, nums: List[int], l: int, r: int) -> int:
        n = len(nums)
        ans = float("inf")
        for i in range(n):
            s = 0
            for j in range(i, n):
                s += nums[j]
                length = j - i + 1
                if length > r:
                    break
                if length >= l and s > 0:
                    ans = min(ans, s)
        return -1 if ans == float("inf") else ans

print(SolutionRunning().minimumSumSubarray([3, -2, 1, 4], 2, 3))   # 1

1


**Độ phức tạp Cách 2:** **Time `O(n * r)`** tệ nhất `O(n^2)` (`n <= 100` nên nhiều nhất `~5000` bước), **Space `O(1)`**.

### Cách 3 — Tổng tiền tố `S[j+1] - S[i]`

**Giải thích.** Dựng `S` một lần (`O(n)`), rồi với mỗi điểm đầu `i` và mỗi **độ dài** `length` từ `l` đến `r` (miễn `i + length <= n`), tổng là `S[i + length] - S[i]`. Cách này **duyệt trực tiếp đúng các độ dài hợp lệ**, không cần `break` hay kiểm tra `length >= l`.

1. `S = list(accumulate(nums, initial=0))`.
2. `for i in range(n)` rồi `for length in range(l, r + 1)`.
3. Nếu `i + length > n` thì mảng con tràn ra ngoài mảng ⇒ `break`.
4. `s = S[i + length] - S[i]`; nếu `s > 0` thì cập nhật `ans`.

In [12]:
from itertools import accumulate

def min_pos_prefix(nums: List[int], l: int, r: int) -> int:
    n = len(nums)
    S = list(accumulate(nums, initial=0))     # S[k] = tổng k phần tử đầu
    inf = float("inf")
    ans = inf
    for i in range(n):
        for length in range(l, r + 1):
            if i + length > n:
                break                         # tràn khỏi mảng
            s = S[i + length] - S[i]
            if s > 0:
                ans = min(ans, s)
    return -1 if ans == inf else ans

for (nums, l, r), expected in examples:
    print(min_pos_prefix(nums, l, r), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
-1 (kỳ vọng -1 )
3 (kỳ vọng 3 )


In [13]:
# Code nộp LeetCode cho Cách 3
from itertools import accumulate
from typing import List

class SolutionPrefix:
    def minimumSumSubarray(self, nums: List[int], l: int, r: int) -> int:
        n = len(nums)
        S = list(accumulate(nums, initial=0))
        ans = float("inf")
        for i in range(n):
            for length in range(l, r + 1):
                if i + length > n:
                    break
                s = S[i + length] - S[i]
                if s > 0:
                    ans = min(ans, s)
        return -1 if ans == float("inf") else ans

print(SolutionPrefix().minimumSumSubarray([1, 2, 3, 4], 2, 4))   # 3

3


**Độ phức tạp Cách 3:** **Time `O(n + n * (r - l + 1))`** ≈ `O(n^2)` tệ nhất, **Space `O(n)`** cho mảng `S`. Ưu điểm: công thức `S[b] - S[a]` dùng được cho **rất nhiều** bài về tổng đoạn; nhược điểm: tốn thêm bộ nhớ so với Cách 2.

### Cách 4 — Cửa sổ trượt (sliding window) cho từng độ dài cố định `k`

**Giải thích.** Với **một độ dài `k` cố định**, mọi mảng con độ dài `k` là các "cửa sổ" cạnh nhau; chuyển từ cửa sổ này sang cửa sổ kế chỉ cần **cộng phần tử mới vào, trừ phần tử cũ ra**. Ta làm lần lượt cho `k = l, l+1, ..., r`.

1. Tính tổng cửa sổ đầu: `w = sum(nums[:k])`; xét `w`.
2. Với mỗi `end` từ `k` đến `n - 1`: `w += nums[end] - nums[end - k]`; xét `w`.
3. "Xét `w`" nghĩa là: nếu `w > 0` thì `ans = min(ans, w)`.

In [14]:
def min_pos_window(nums: List[int], l: int, r: int) -> int:
    n = len(nums)
    inf = float("inf")
    ans = inf
    for k in range(l, r + 1):
        w = sum(nums[:k])                  # cửa sổ đầu tiên
        if w > 0:
            ans = min(ans, w)
        for end in range(k, n):
            w += nums[end] - nums[end - k]   # thêm đầu mới, bỏ đuôi cũ
            if w > 0:
                ans = min(ans, w)
    return -1 if ans == inf else ans

for (nums, l, r), expected in examples:
    print(min_pos_window(nums, l, r), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
-1 (kỳ vọng -1 )
3 (kỳ vọng 3 )


In [15]:
# Code nộp LeetCode cho Cách 4
from typing import List

class SolutionWindow:
    def minimumSumSubarray(self, nums: List[int], l: int, r: int) -> int:
        n = len(nums)
        ans = float("inf")
        for k in range(l, r + 1):
            w = sum(nums[:k])
            if w > 0:
                ans = min(ans, w)
            for end in range(k, n):
                w += nums[end] - nums[end - k]
                if w > 0:
                    ans = min(ans, w)
        return -1 if ans == float("inf") else ans

print(SolutionWindow().minimumSumSubarray([-2, 2, -3, 1], 2, 3))   # -1

-1


**Độ phức tạp Cách 4:** **Time `O(n * (r - l + 1))`** (mỗi `k` tốn `O(n)`), **Space `O(1)`**. Cách này hợp khi đề hỏi "mọi mảng con có độ dài đúng `k`" — mẫu rất hay gặp.

**Cả bốn cách có cùng đáp án**; khác nhau ở độ sạch và khả năng mở rộng. Với `n <= 100`, cách nào cũng qua; trong phỏng vấn nên chọn Cách 2 hoặc Cách 3 và giải thích được vì sao nhanh hơn vét cạn.

## 8. TRACE từng bước

Trace **Cách 2** với `nums = [3, -2, 1, 4]`, `l = 2`, `r = 3`. Cột `ans` là đáp án **sau** bước đó.

| i | j | nums[j] | s (sau cộng) | length | hành động | ans |
|---|---|---|---|---|---|---|
| 0 | 0 | 3 | 3 | 1 | length < l: bỏ qua | inf |
| 0 | 1 | -2 | 1 | 2 | hợp lệ, s>0: cập nhật | **1** |
| 0 | 2 | 1 | 2 | 3 | hợp lệ, s>0: `min(1, 2)` | 1 |
| 0 | 3 | 4 | 6 | 4 | length > r: **break** | 1 |
| 1 | 1 | -2 | -2 | 1 | length < l | 1 |
| 1 | 2 | 1 | -1 | 2 | hợp lệ nhưng s<=0: bỏ qua | 1 |
| 1 | 3 | 4 | 3 | 3 | hợp lệ, s>0: `min(1, 3)` | 1 |
| 2 | 2 | 1 | 1 | 1 | length < l | 1 |
| 2 | 3 | 4 | 5 | 2 | hợp lệ, s>0: `min(1, 5)` | 1 |
| 3 | 3 | 4 | 4 | 1 | length < l | 1 |

Kết quả `1`. Dòng `(0,3)` cho thấy `break`; dòng `(1,2)` cho thấy tổng `-1` bị loại mà vòng lặp **vẫn tiếp tục** sang `j = 3`.

In [16]:
def trace_min_pos(nums, l, r):
    n = len(nums)
    inf = float("inf")
    ans = inf
    print(f"nums={nums}, l={l}, r={r}")
    for i in range(n):
        s = 0
        for j in range(i, n):
            s += nums[j]
            length = j - i + 1
            if length > r:
                print(f"i={i} j={j} nums[j]={nums[j]:>2} s={s:>3} length={length} -> dài hơn r: break")
                break
            if length < l:
                action = "length < l: bỏ qua"
            elif s <= 0:
                action = "tổng không dương: bỏ qua"
            else:
                old = ans
                ans = min(ans, s)
                action = f"cập nhật ans: min({old}, {s}) = {ans}"
            print(f"i={i} j={j} nums[j]={nums[j]:>2} s={s:>3} length={length} -> {action}")
    result = -1 if ans == inf else ans
    print("kết quả:", result)
    return result

trace_min_pos([3, -2, 1, 4], 2, 3)

nums=[3, -2, 1, 4], l=2, r=3
i=0 j=0 nums[j]= 3 s=  3 length=1 -> length < l: bỏ qua
i=0 j=1 nums[j]=-2 s=  1 length=2 -> cập nhật ans: min(inf, 1) = 1
i=0 j=2 nums[j]= 1 s=  2 length=3 -> cập nhật ans: min(1, 2) = 1
i=0 j=3 nums[j]= 4 s=  6 length=4 -> dài hơn r: break
i=1 j=1 nums[j]=-2 s= -2 length=1 -> length < l: bỏ qua
i=1 j=2 nums[j]= 1 s= -1 length=2 -> tổng không dương: bỏ qua
i=1 j=3 nums[j]= 4 s=  3 length=3 -> cập nhật ans: min(1, 3) = 1
i=2 j=2 nums[j]= 1 s=  1 length=1 -> length < l: bỏ qua
i=2 j=3 nums[j]= 4 s=  5 length=2 -> cập nhật ans: min(1, 5) = 1
i=3 j=3 nums[j]= 4 s=  4 length=1 -> length < l: bỏ qua
kết quả: 1


1

In [17]:
# Trace thứ hai: ví dụ không có đáp án
trace_min_pos([-2, 2, -3, 1], 2, 3)

nums=[-2, 2, -3, 1], l=2, r=3
i=0 j=0 nums[j]=-2 s= -2 length=1 -> length < l: bỏ qua
i=0 j=1 nums[j]= 2 s=  0 length=2 -> tổng không dương: bỏ qua
i=0 j=2 nums[j]=-3 s= -3 length=3 -> tổng không dương: bỏ qua
i=0 j=3 nums[j]= 1 s= -2 length=4 -> dài hơn r: break
i=1 j=1 nums[j]= 2 s=  2 length=1 -> length < l: bỏ qua
i=1 j=2 nums[j]=-3 s= -1 length=2 -> tổng không dương: bỏ qua
i=1 j=3 nums[j]= 1 s=  0 length=3 -> tổng không dương: bỏ qua
i=2 j=2 nums[j]=-3 s= -3 length=1 -> length < l: bỏ qua
i=2 j=3 nums[j]= 1 s= -2 length=2 -> tổng không dương: bỏ qua
i=3 j=3 nums[j]= 1 s=  1 length=1 -> length < l: bỏ qua
kết quả: -1


-1

## 9. Code nộp LeetCode

Ta nộp **Cách 2** (đơn giản, `O(1)` bộ nhớ). Trên LeetCode, `List` đã được import sẵn; trong notebook ta tự import.

In [18]:
from typing import List

class Solution:
    def minimumSumSubarray(self, nums: List[int], l: int, r: int) -> int:
        n = len(nums)
        inf = float("inf")
        ans = inf
        for i in range(n):
            s = 0                                   # tổng của nums[i..j]
            for j in range(i, n):
                s += nums[j]
                length = j - i + 1
                if length > r:
                    break                           # từ đây độ dài chỉ tăng thêm
                if length >= l and s > 0:
                    ans = min(ans, s)
        return -1 if ans == inf else ans


for (nums, l, r), expected in examples:
    got = Solution().minimumSumSubarray(nums, l, r)
    print(nums, l, r, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[3, -2, 1, 4] 2 3 -> 1 OK
[-2, 2, -3, 1] 2 3 -> -1 OK
[1, 2, 3, 4] 2 4 -> 3 OK


## 10. Vì sao thuật toán đúng

**Bất biến.** Khi vòng lặp xét cặp `(i, j)`, biến `s` đúng bằng `nums[i] + ... + nums[j]`. Thật vậy, đầu mỗi vòng `i`, `s = 0` (tổng rỗng); mỗi lần `j` tăng, ta cộng đúng `nums[j]`.

**Không bỏ sót.** Mọi mảng con độ dài `[l, r]` ứng với đúng một cặp `(i, j)` có `l <= j - i + 1 <= r`. Vòng lặp duyệt `i` từ `0..n-1`, `j` từ `i` đến khi độ dài vượt `r` — tức là **đủ** các cặp có độ dài `<= r`; điều kiện `length >= l` lọc phần còn lại.

**Dừng sớm an toàn.** Với `i` cố định, độ dài `j - i + 1` tăng theo `j`; một khi `> r`, mọi `j` lớn hơn cũng `> r`, nên `break` không làm mất mảng con hợp lệ nào.

**Không trả nhầm.** Ta chỉ cập nhật `ans` ở những mảng con thoả **cả** `length` hợp lệ **và** `s > 0`, và lấy `min`. Nếu không có mảng con nào thoả thì `ans` giữ nguyên `inf` ⇒ trả `-1` (đúng đề).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

- **Cách 1 (ba vòng):** Time `O(n^3)`, Space `O(1)` (hoặc `O(n)` nếu cắt lát).
- **Cách 2 (cộng dồn + break):** Time `O(n * min(n, r))` ⇒ tệ nhất `O(n^2)`, Space `O(1)`.
- **Cách 3 (tổng tiền tố):** Time `O(n + n * (r - l + 1))` ⇒ `O(n^2)`, Space `O(n)`.
- **Cách 4 (cửa sổ trượt cho từng `k`):** Time `O(n * (r - l + 1))`, Space `O(1)`.

Với `n <= 100`: số bước tối đa khoảng `100 * 100 = 10^4` — rất nhỏ. Nếu `n` lên `10^5` thì cần ý tưởng khác (ví dụ cấu trúc dữ liệu có thứ tự trên tổng tiền tố để tìm "tổng dương nhỏ nhất"), nằm ngoài phạm vi bài Easy này.

## 12. Bộ test

### 12.1. Ví dụ của đề và các trường hợp biên

Với các cách `Solution`, `SolutionPrefix`, `SolutionWindow`, `min_pos_brute`, ta cùng kiểm tra. Các test "tự dựng" ghi rõ trong chú thích; kết quả đã tính tay trong notebook này.

In [19]:
impls = {
    "brute":   min_pos_brute,
    "running": lambda a, l, r: Solution().minimumSumSubarray(a, l, r),
    "prefix":  lambda a, l, r: SolutionPrefix().minimumSumSubarray(a, l, r),
    "window":  lambda a, l, r: SolutionWindow().minimumSumSubarray(a, l, r),
}

cases = [
    # (nums, l, r, kỳ vọng)
    ([3, -2, 1, 4], 2, 3, 1),        # ví dụ 1
    ([-2, 2, -3, 1], 2, 3, -1),      # ví dụ 2
    ([1, 2, 3, 4], 2, 4, 3),         # ví dụ 3
    ([5], 1, 1, 5),                  # tự dựng: một phần tử dương
    ([-5], 1, 1, -1),                # tự tạo: một phần tử âm
    ([0], 1, 1, -1),                 # tự dựng: tổng 0 không phải dương
    ([0, 0, 0], 1, 3, -1),           # tự dựng: toàn 0
    ([1, -1, 1, -1], 2, 2, -1),      # tự dựng: mọi cặp đều có tổng 0
    ([2, -1, 2], 1, 3, 1),           # tự dựng: [2,-1]=1
    ([-1, -1, 3], 1, 3, 1),          # tự dựng: [-1,-1,3] = 1
    ([1000] * 5, 5, 5, 5000),        # tự dựng: chỉ có đúng một cửa sổ
    ([-1000] * 4 + [1000], 1, 5, 1000),  # tự dựng: chỉ [1000] có tổng dương
]

for nums, l, r, exp in cases:
    for name, f in impls.items():
        got = f(nums, l, r)
        assert got == exp, (name, nums, l, r, got, exp)
print("12.1 OK:", len(cases), "test x", len(impls), "cách")

12.1 OK: 12 test x 4 cách


Lưu ý với test cuối: `[-1000, -1000, -1000, -1000, 1000]` với `l = 1, r = 5`: các tổng dương duy nhất là mảng con chỉ chứa phần tử cuối (`[1000]` = 1000); mọi mảng con còn lại đều có tổng âm. Vậy đáp án `1000`.

### 12.2. Phản ví dụ cho các cách làm sai

Ta viết **bản sai có chủ đích** rồi tìm input làm lộ lỗi. Đây là cách tự kiểm tra rằng test của ta đủ mạnh.

In [20]:
def wrong_ge0(nums, l, r):
    """Sai: coi tổng 0 là hợp lệ (s >= 0)."""
    ans = float("inf")
    for i in range(len(nums)):
        s = 0
        for j in range(i, len(nums)):
            s += nums[j]
            if l <= j - i + 1 <= r and s >= 0:
                ans = min(ans, s)
    return -1 if ans == float("inf") else ans

def wrong_nolen(nums, l, r):
    """Sai: quên kiểm tra độ dài."""
    ans = float("inf")
    for i in range(len(nums)):
        s = 0
        for j in range(i, len(nums)):
            s += nums[j]
            if s > 0:
                ans = min(ans, s)
    return -1 if ans == float("inf") else ans

def wrong_init0(nums, l, r):
    """Sai: khởi tạo ans = 0."""
    ans = 0
    for i in range(len(nums)):
        s = 0
        for j in range(i, len(nums)):
            s += nums[j]
            if l <= j - i + 1 <= r and s > 0:
                ans = min(ans, s)
    return ans

print("ge0   :", wrong_ge0([0], 1, 1), "(đúng phải -1)")
print("nolen :", wrong_nolen([1, 5], 2, 2), "(đúng phải 6)")
print("init0 :", wrong_init0([3, -2, 1, 4], 2, 3), "(đúng phải 1)")
assert wrong_ge0([0], 1, 1) == 0 and min_pos_brute([0], 1, 1) == -1
assert wrong_nolen([1, 5], 2, 2) == 1 and min_pos_brute([1, 5], 2, 2) == 6
assert wrong_init0([3, -2, 1, 4], 2, 3) == 0 and min_pos_brute([3, -2, 1, 4], 2, 3) == 1
print("12.2 OK: ba bản sai đều bị lộ")

ge0   : 0 (đúng phải -1)
nolen : 1 (đúng phải 6)
init0 : 0 (đúng phải 1)
12.2 OK: ba bản sai đều bị lộ


### 12.3. Đối chiếu ngẫu nhiên (có seed) và vét cạn độc lập

Ta so **bốn cách** với một **oracle độc lập** (liệt kê mọi mảng con bằng cắt lát và dùng `min` trên danh sách) trên nhiều input ngẫu nhiên đúng ràng buộc: `1 <= n <= 100`, `1 <= l <= r <= n`, `-1000 <= nums[i] <= 1000`. Dùng cả giá trị nhỏ (nhiều tổng `0` và trùng) lẫn giá trị lớn.

In [21]:
import random

def oracle(nums, l, r):
    sums = []
    n = len(nums)
    for a in range(n):
        for b in range(a + 1, n + 1):
            if l <= b - a <= r and sum(nums[a:b]) > 0:
                sums.append(sum(nums[a:b]))
    return min(sums) if sums else -1

random.seed(3364)
count = 0
for _ in range(1500):
    n = random.randint(1, 12)
    lim = random.choice([2, 5, 1000])
    nums = [random.randint(-lim, lim) for _ in range(n)]
    l = random.randint(1, n)
    r = random.randint(l, n)
    exp = oracle(nums, l, r)
    for name, f in impls.items():
        assert f(nums, l, r) == exp, (name, nums, l, r)
    count += 1

# vài input lớn đúng ràng buộc (n = 100)
for _ in range(60):
    n = 100
    nums = [random.randint(-1000, 1000) for _ in range(n)]
    l = random.randint(1, n)
    r = random.randint(l, n)
    exp = oracle(nums, l, r)
    for name, f in impls.items():
        assert f(nums, l, r) == exp
    count += 1
print("12.3 OK:", count, "input ngẫu nhiên, 4 cách khớp oracle")

12.3 OK: 1560 input ngẫu nhiên, 4 cách khớp oracle


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `s >= 0` thay vì `s > 0`

Đề yêu cầu tổng **lớn hơn 0**; tổng `0` **không** hợp lệ. Dùng `>=` sẽ trả `0` ở các input như `[0]` hay `[1, -1]` (đáng lẽ `-1`).

In [22]:
# SAI
def sai_1(nums, l, r):
    ans = float("inf")
    for i in range(len(nums)):
        s = 0
        for j in range(i, len(nums)):
            s += nums[j]
            if l <= j - i + 1 <= r and s >= 0:     # <-- sai: nhận cả tổng 0
                ans = min(ans, s)
    return -1 if ans == float("inf") else ans

print(sai_1([1, -1], 2, 2))   # 0  (sai: đề yêu cầu tổng > 0 nên đáp án là -1)

# ĐÚNG: s > 0
print(Solution().minimumSumSubarray([1, -1], 2, 2))   # -1

0
-1


### Lỗi 2 — Quên kiểm tra độ dài `l..r`

Chỉ `min` của mọi tổng dương là sai vì có thể chọn mảng con **quá ngắn** (hoặc quá dài). Ví dụ `[1, 5]`, `l = r = 2` — đáp án `6`, không phải `1`.

In [23]:
# SAI
def sai_2(nums, l, r):
    ans = float("inf")
    for i in range(len(nums)):
        s = 0
        for j in range(i, len(nums)):
            s += nums[j]
            if s > 0:                             # <-- sai: thiếu kiểm tra độ dài
                ans = min(ans, s)
    return -1 if ans == float("inf") else ans

print(sai_2([1, 5], 2, 2))                       # 1  (sai, vì [1] có độ dài 1 < l)
print(Solution().minimumSumSubarray([1, 5], 2, 2))   # 6 (đúng)

1
6


### Lỗi 3 — Khởi tạo `ans = 0` hoặc trả `inf` ra ngoài

`ans = 0` làm `min(ans, s)` không bao giờ nhận tổng dương. Quên đổi `inf` thành `-1` thì khi không có đáp án ta trả `inf` (hoặc một số khổng lồ) thay vì `-1`.

In [24]:
# SAI 1: ans = 0
def sai_3a(nums, l, r):
    ans = 0                                       # <-- sai
    for i in range(len(nums)):
        s = 0
        for j in range(i, len(nums)):
            s += nums[j]
            if l <= j - i + 1 <= r and s > 0:
                ans = min(ans, s)
    return ans

# SAI 2: quên đổi inf thành -1
def sai_3b(nums, l, r):
    ans = float("inf")
    for i in range(len(nums)):
        s = 0
        for j in range(i, len(nums)):
            s += nums[j]
            if l <= j - i + 1 <= r and s > 0:
                ans = min(ans, s)
    return ans                                    # <-- sai: có thể trả inf

print(sai_3a([3, -2, 1, 4], 2, 3))               # 0 (sai; đúng là 1)
print(sai_3b([-2, 2, -3, 1], 2, 3))              # inf (sai; đúng là -1)
print(Solution().minimumSumSubarray([-2, 2, -3, 1], 2, 3))   # -1 (đúng)

0
inf
-1


### Lỗi 4 — Cắt lát thiếu `+ 1` (off-by-one)

`nums[i:j]` **không gồm `nums[j]`**. Muốn mảng con `i..j` phải viết `nums[i:j + 1]`. Với `[1, 5]`, `i = 0, j = 1`: `nums[0:1] = [1]`, trong khi `nums[0:2] = [1, 5]`.

In [25]:
nums = [1, 5]
i, j = 0, 1
print(sum(nums[i:j]))        # 1 (sai: thiếu phần tử j)
print(sum(nums[i:j + 1]))    # 6 (đúng)
print(j - i + 1)             # 2 -> độ dài thật của nums[i..j]

1
6
2


### Lỗi 5 — `break` quá sớm khi tổng không dương

Một số người thấy `s <= 0` thì `break`, nghĩ rằng "đã hỏng". Nhưng mảng có số **âm** lẫn **dương** nên mở rộng thêm có thể làm tổng dương lên lại. Chỉ được `break` khi **độ dài vượt `r`**.

In [26]:
def sai_5(nums, l, r):
    ans = float("inf")
    for i in range(len(nums)):
        s = 0
        for j in range(i, len(nums)):
            s += nums[j]
            if s <= 0:
                break                            # <-- sai
            if l <= j - i + 1 <= r:
                ans = min(ans, s)
    return -1 if ans == float("inf") else ans

print(sai_5([-2, 1, 4], 2, 3))                  # -1 (sai: [-2,1,4] = 3 > 0 bị bỏ sót)
print(Solution().minimumSumSubarray([-2, 1, 4], 2, 3))   # 3 (đúng)

5
3


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có cụm **"subarray + điều kiện độ dài + tổng"** và `n` nhỏ ⇒ nghĩ ngay **liệt kê cặp `(i, j)` kèm tổng chạy** `O(n^2)`.
- Cần tổng đoạn nhiều lần ⇒ **prefix sum**: `S[j+1] - S[i]`.
- Độ dài **cố định** ⇒ **sliding window** (cộng phần tử mới, trừ phần tử cũ).
- Bài "nhỏ nhất/lớn nhất trong các đối tượng thoả điều kiện" ⇒ **lọc rồi `min/max`** với giá trị lính canh (`inf` / `-inf`), cuối cùng đổi sang đáp án "không có" (`-1`).
- Ràng buộc `n <= 100` là gợi ý đề **cho phép `O(n^2)` hay thậm chí `O(n^3)`**.
- Luôn tự hỏi các điều kiện biên: tổng bằng `0`, độ dài đúng bằng `l` hoặc `r`, `l = r`, `n = 1`, tất cả âm.
- Khi đọc đề, **gạch chân** các điều kiện: "size between l and r **inclusive**", "sum **greater than 0**" — hai từ khoá này là nơi sinh lỗi nhiều nhất.

## 15. Biến thể của bài

- **Tổng dương nhỏ nhất, không giới hạn độ dài:** thử mọi mảng con (mục 12 oracle) — cùng cách, bỏ điều kiện độ dài.
- **Tổng lớn nhất** với độ dài `[l, r]` (có thể âm): đổi `min` thành `max`, lính canh `-inf`.
- **Đếm** số mảng con có độ dài `[l, r]` và tổng dương: thay `ans = min(...)` bằng `count += 1` (bài tập 4).
- **Tổng nhỏ nhất không âm (`>= 0`)**: đổi điều kiện thành `s >= 0`.
- **Tổng nhỏ nhất `>= K`**: đổi `s > 0` thành `s >= K` (tổng quát hoá).
- **Trả cả vị trí** mảng con tối ưu (bài tập 5).
- **Mảng lớn hơn (`n` đến `10^5`)** với cùng đề: cần tổng tiền tố và cấu trúc có thứ tự (sorted container / cây cân bằng) để tìm `S[j+1] - S[i]` dương nhỏ nhất; ngoài phạm vi Easy.
- **Mảng con có tổng dương nhỏ nhất nhưng độ dài tối thiểu `l` và tối đa `n`**: đặt `r = n`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn ba vòng | Với mỗi `(i, j)` gọi `sum(nums[i:j+1])` | `O(n^3)` | `O(1)`/`O(n)` | Để kiểm chứng; chấp nhận được khi `n` rất nhỏ |
| 2. Cộng dồn + break | Cố định `i`, mở rộng `j`, `s += nums[j]` | `O(n^2)` | `O(1)` | **Cách nộp chính**: đơn giản, ít bộ nhớ |
| 3. Tổng tiền tố | `S[i+len] - S[i]` cho mỗi `len` trong `[l, r]` | `O(n^2)` | `O(n)` | Khi cần tổng đoạn nhiều lần, nhiều truy vấn |
| 4. Cửa sổ trượt | Với mỗi `k`, trượt cửa sổ độ dài `k` | `O(n * (r-l+1))` | `O(1)` | Khi đề xoay quanh độ dài cố định `k` |

Tất cả đều cho cùng đáp án (đã đối chiếu ở mục 12.3).

## 17. 5 bài tập vận dụng (dễ → khó)

Các cell TODO **không được gọi khi chạy notebook** — hãy tự điền rồi tự chạy. Đáp án ở mục 19.

### Bài 1 — Đoán output

Đoán kết quả của đoạn code sau (không chạy). Gợi ý: liệt kê tay các mảng con độ dài 1 và 2.

In [27]:
def bai1():
    nums, l, r = [2, -1, 2], 1, 2
    out = []
    for i in range(len(nums)):
        s = 0
        for j in range(i, len(nums)):
            s += nums[j]
            if l <= j - i + 1 <= r and s > 0:
                out.append(s)
    return sorted(out), (min(out) if out else -1)

# Hãy đoán rồi mới chạy:  print(bai1())

### Bài 2 — Tính tay

Cho `nums = [-3, 4, -2, 5, -1]`, `l = 2`, `r = 3`. Hãy viết mảng `S` (tổng tiền tố) rồi tính tay đáp án bằng công thức `S[i + length] - S[i]`. Điền hai số vào hai biến dưới đây (không chạy hàm để lấy đáp án).

In [28]:
# TODO: tự tính tay rồi điền
bai2_S = None        # mảng tổng tiền tố, độ dài 6
bai2_ans = None      # đáp án của bài

### Bài 3 — Điền TODO

Hoàn thành hàm theo Cách 3 (tổng tiền tố). Hàm trả về tổng dương nhỏ nhất của mảng con độ dài trong `[l, r]`, hoặc `-1`.

In [29]:
def bai3(nums, l, r):
    from itertools import accumulate
    S = list(accumulate(nums, initial=0))
    ans = float("inf")
    n = len(nums)
    # TODO: duyệt i và length; tính s = S[i + length] - S[i]; nếu s > 0 thì cập nhật ans
    ...
    # TODO: trả -1 nếu ans vẫn là vô cực
    ...

### Bài 4 — Tự viết lại

**Đếm** số mảng con có độ dài trong `[l, r]` và tổng **lớn hơn 0**. Tự viết hàm `bai4(nums, l, r)`; ví dụ `[3, -2, 1, 4], 2, 3` có 4 mảng con như vậy (xem mục 2).

In [30]:
def bai4(nums, l, r):
    # TODO: tự viết, trả về số mảng con
    ...

### Bài 5 — Biến thể / mini challenge

Trả về bộ ba `(tổng, i, j)` của mảng con có tổng dương nhỏ nhất với độ dài trong `[l, r]`. Nếu có nhiều mảng con cùng tổng nhỏ nhất, chọn cái có `i` nhỏ nhất, rồi `j` nhỏ nhất. Không có thì trả `(-1, -1, -1)`.

Ví dụ: `[3, -2, 1, 4], 2, 3` → `(1, 0, 1)`.

In [31]:
def bai5(nums, l, r):
    # TODO: tự viết
    ...

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.**
- Mức 1: liệt kê các mảng con có độ dài 1 và 2 của `[2, -1, 2]`.
- Mức 2: độ dài 1 cho `2, -1, 2` (loại `-1`), độ dài 2 cho `[2, -1] = 1`, `[-1, 2] = 1`.
- Mức 3: danh sách các tổng hợp lệ (đã sắp xếp) là `[1, 1, 2, 2]`, nhỏ nhất `1`.

**Bài 2.**
- Mức 1: `S[0] = 0`, `S[k+1] = S[k] + nums[k]`.
- Mức 2: `S = [0, -3, 1, -1, 4, 3]`.
- Mức 3: độ dài 2 cho tổng `1, 2, 3, 4`; độ dài 3 cho `-1, 7, 2`; tổng dương nhỏ nhất là `1`.

**Bài 3.**
- Mức 1: `for i in range(n)` rồi `for length in range(l, r + 1)`.
- Mức 2: nếu `i + length > n` thì `break`.
- Mức 3: `s = S[i + length] - S[i]`; nếu `s > 0` thì `ans = min(ans, s)`; cuối cùng `return -1 if ans == float("inf") else ans`.

**Bài 4.**
- Mức 1: giống Cách 2 nhưng không cần `min`.
- Mức 2: thay `ans = min(ans, s)` bằng `dem += 1`.
- Mức 3: khởi tạo `dem = 0`; chỉ đếm khi `length >= l` và `s > 0`; vẫn `break` khi `length > r`.

**Bài 5.**
- Mức 1: lưu thêm `i, j` khi cập nhật.
- Mức 2: duyệt `i` tăng dần, `j` tăng dần; chỉ cập nhật khi `s < best` (**nghiêm ngặt**), để cái xuất hiện trước thắng khi hoà.
- Mức 3: trả `(-1, -1, -1)` nếu `best` vẫn là `inf`.

## 19. Đáp án tham khảo cho bài tập

Mỗi đáp án đều được kiểm tra bằng `assert`.

In [32]:
# --- Bài 1 ---
assert bai1() == ([1, 1, 2, 2], 1)
print("Bài 1:", bai1())

# --- Bài 2 ---
bai2_S_dap_an = [0, -3, 1, -1, 4, 3]
bai2_ans_dap_an = 1
from itertools import accumulate
assert list(accumulate([-3, 4, -2, 5, -1], initial=0)) == bai2_S_dap_an
assert Solution().minimumSumSubarray([-3, 4, -2, 5, -1], 2, 3) == bai2_ans_dap_an
print("Bài 2:", bai2_S_dap_an, bai2_ans_dap_an)

# --- Bài 3 ---
def bai3_dap_an(nums, l, r):
    from itertools import accumulate
    S = list(accumulate(nums, initial=0))
    ans = float("inf")
    n = len(nums)
    for i in range(n):
        for length in range(l, r + 1):
            if i + length > n:
                break
            s = S[i + length] - S[i]
            if s > 0:
                ans = min(ans, s)
    return -1 if ans == float("inf") else ans

assert bai3_dap_an([3, -2, 1, 4], 2, 3) == 1
assert bai3_dap_an([-2, 2, -3, 1], 2, 3) == -1
assert bai3_dap_an([1, 2, 3, 4], 2, 4) == 3
print("Bài 3 OK")

# --- Bài 4 ---
def bai4_dap_an(nums, l, r):
    dem = 0
    for i in range(len(nums)):
        s = 0
        for j in range(i, len(nums)):
            s += nums[j]
            length = j - i + 1
            if length > r:
                break
            if length >= l and s > 0:
                dem += 1
    return dem

assert bai4_dap_an([3, -2, 1, 4], 2, 3) == 4
assert bai4_dap_an([-2, 2, -3, 1], 2, 3) == 0           # ví dụ 2: không có mảng con nào
assert bai4_dap_an([1, 2, 3, 4], 2, 4) == 6              # 3 + 2 + 1 mảng con độ dài 2,3,4
print("Bài 4:", bai4_dap_an([3, -2, 1, 4], 2, 3))

# --- Bài 5 ---
def bai5_dap_an(nums, l, r):
    best = float("inf")
    bi = bj = -1
    for i in range(len(nums)):
        s = 0
        for j in range(i, len(nums)):
            s += nums[j]
            length = j - i + 1
            if length > r:
                break
            if length >= l and 0 < s < best:      # nghiêm ngặt: cái xuất hiện trước thắng
                best, bi, bj = s, i, j
    return (-1, -1, -1) if best == float("inf") else (best, bi, bj)

assert bai5_dap_an([3, -2, 1, 4], 2, 3) == (1, 0, 1)
assert bai5_dap_an([-2, 2, -3, 1], 2, 3) == (-1, -1, -1)
assert bai5_dap_an([1, 2, 3, 4], 2, 4) == (3, 0, 1)
assert bai5_dap_an([2, -1, 2], 1, 2) == (1, 0, 1)
print("Bài 5 OK")

Bài 1: ([1, 1, 2, 2], 1)
Bài 2: [0, -3, 1, -1, 4, 3] 1
Bài 3 OK
Bài 4: 4
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (cộng dồn khi mở rộng đầu phải):**

```python
class Solution:
    def minimumSumSubarray(self, nums, l, r):
        n = len(nums)
        ans = float("inf")
        for i in range(n):
            s = 0
            for j in range(i, n):
                s += nums[j]
                length = j - i + 1
                if length > r:
                    break
                if length >= l and s > 0:
                    ans = min(ans, s)
        return -1 if ans == float("inf") else ans
```

**Checklist trước khi submit:**

- [ ] Tên method đúng: `minimumSumSubarray(self, nums, l, r)`.
- [ ] Điều kiện tổng là `s > 0` (không phải `>= 0`).
- [ ] Độ dài trong `[l, r]` **gồm cả hai đầu mút**.
- [ ] `ans` khởi tạo bằng `inf`; cuối cùng đổi `inf` thành `-1`.
- [ ] `s = 0` được **đặt lại** mỗi lần `i` mới.
- [ ] Không `break` khi tổng không dương; chỉ `break` khi độ dài vượt `r`.
- [ ] Đã thử: ví dụ 1, 2, 3; `[0]`; `l = r`; mảng toàn âm; mảng `n = 100`.